# COVID-BLUES Verification

In [ ]:
from pathlib import Path

import imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import skimage

RAW_DATA_DIR = Path("/scratch/data/covid_blues_raw_20260523/covid_blues/")

In [ ]:
# Create a data frame with "patient", "blue_point" and "video_path" extracted
# from the video paths


def parse_file_path(path):
    # Example filename: "patient_{patient}_{blue_point}.mp4"
    parts = path.stem.split("_")
    return {"patient": parts[1], "blue_point": parts[2], "video": path}


video_paths = list(RAW_DATA_DIR.glob("lus_videos/*.mp4"))
video_data = pd.DataFrame([parse_file_path(p) for p in video_paths])
video_data.head()

In [ ]:
# Load the clinical variables from the CSV file
clinical_data = pd.read_csv(RAW_DATA_DIR / "clinical_variables.csv")
clinical_data.head()

In [ ]:
for col in clinical_data.columns:
    print(col)

In [ ]:
# Filter the clinical data to only include the following columns:
# - patient_id
# - cov_test
# - clin_diagn#healthy_lung
# - clin_diagn#bacterial_pneumonia
# - clin_diagn#COVID19_pneumonia
# - clin_diagn#other_viral_pneumonia
# - clin_diagn#other_lung_disease
columns_to_keep = [
    "patient_id",
    "cov_test",
    "clin_diagn#healthy_lung",
    "clin_diagn#bacterial_pneumonia",
    "clin_diagn#COVID19_pneumonia",
    "clin_diagn#other_viral_pneumonia",
    "clin_diagn#other_lung_disease",
]

clinical_data_filtered = clinical_data[columns_to_keep].copy()
clinical_data_filtered.head()

In [ ]:
# Check that only one of the "clin_diagn" columns is 1 for each patient
diagnosis_cols = [
    "clin_diagn#healthy_lung",
    "clin_diagn#bacterial_pneumonia",
    "clin_diagn#COVID19_pneumonia",
    "clin_diagn#other_viral_pneumonia",
    "clin_diagn#other_lung_disease",
]

clinical_data_filtered["diagnosis_sum"] = clinical_data_filtered[diagnosis_cols].sum(
    axis=1
)
clinical_data_filtered["diagnosis_sum"].value_counts()

In [ ]:
# Check that whenever "cov_test" is 1, "clin_diagn#COVID19_pneumonia" is also 1
cov_test_positive = clinical_data_filtered[clinical_data_filtered["cov_test"] == 1]
cov_test_positive["clin_diagn#COVID19_pneumonia"].value_counts()

In [ ]:
# Create a label column with either ("healthy_lung", "bacterial_pneumonia",
# "COVID19_pneumonia", "other_viral_pneumonia", "other_lung_disease") based on
# the "clin_diagn#" columns
def get_label(row):
    if row["clin_diagn#healthy_lung"] == 1:
        return "healthy_lung"
    elif row["clin_diagn#bacterial_pneumonia"] == 1:
        return "bacterial_pneumonia"
    elif row["clin_diagn#COVID19_pneumonia"] == 1:
        return "COVID19_pneumonia"
    elif row["clin_diagn#other_viral_pneumonia"] == 1:
        return "other_viral_pneumonia"
    elif row["clin_diagn#other_lung_disease"] == 1:
        return "other_lung_disease"
    else:
        return "unknown"


clinical_data_filtered["label"] = clinical_data_filtered.apply(get_label, axis=1)
clinical_data_filtered.head()

In [ ]:
# Plot the distribution of "label"
sns.countplot(data=clinical_data_filtered, x="label")
plt.xticks(rotation=45)
plt.title("Distribution of Clinical Diagnoses")
plt.xlabel("Clinical Diagnosis")
plt.ylabel("Count")
plt.show()

In [ ]:
# Extract the frames of a video using imageio
video = imageio.get_reader(video_data["video"].iloc[0])

frames = []
for i, frame in enumerate(video):  # type: ignore
    frames.append(frame)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i in range(5):
    axes[i].imshow(frames[i])
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")

plt.show()

In [ ]:
# Create a scan mask for the video by looking at the pixels that change value
# across frames. This will give us a binary mask of the area that was scanned.
frame_array = np.array(frames)  # (frames, height, width, channels)

grayscale_frames = np.mean(frame_array, axis=-1)  # (frames, height, width)
pixel_variance = np.var(grayscale_frames, axis=0)  # (height, width)

scan_mask = pixel_variance > 0  # Binary mask of scanned area
scan_mask = skimage.morphology.convex_hull_image(scan_mask)


plt.imshow(scan_mask, cmap="gray")
plt.show()

In [ ]:
# Add the height, width, number of channels and number of frames to the video
# data frame


def get_video_metadata(video_path):
    video = imageio.get_reader(video_path)
    frames = []
    for i, frame in enumerate(video):  # type: ignore
        frames.append(frame)
    frame_array = np.array(frames)  # (frames, height, width, channels)
    return {
        "num_frames": frame_array.shape[0],
        "height": frame_array.shape[1],
        "width": frame_array.shape[2],
        "num_channels": frame_array.shape[3],
    }


video_data[["num_frames", "height", "width", "num_channels"]] = video_data[
    "video"
].apply(lambda x: pd.Series(get_video_metadata(x)))
video_data.head()

In [ ]:
video_data[["num_frames", "height", "width", "num_channels"]].describe()

In [ ]:
# Count the number of videos and total number of frames
len(video_data), video_data["num_frames"].sum()